In [ ]:
# Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Install Hugging Face transformers
!pip install -q transformers

In [ ]:
import os
import torch

# Confirm GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Using device: {device}")
if torch.cuda.is_available():
  print(f"[+] GPU Model: {torch.cuda.get_device_name(0)}")

# Define path to your project folder in Google Drive
DRIVE_DIR = "/content/drive/MyDrive/neuropeptide-classifier-benchmark/Data"
OUTPUT_DIR = os.path.join(DRIVE_DIR, "processed_tensors")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify source CSV files exist
train_csv = os.path.join(DRIVE_DIR, "train.csv")
test_csv = os.path.join(DRIVE_DIR, "test.csv")

assert os.path.exists(train_csv), f"Could not find {train_csv}"
assert os.path.exists(test_csv), f"Could not find {test_csv}"
print("[+] Project folder and source CSV files verified successfully!")

[+] Using device: cuda
[+] GPU Model: Tesla T4
[+] Project folder and source CSV files verified successfully!


In [ ]:
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "facebook/esm2_t33_650M_UR50D"

print(f"[+] Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)

# Freeze ESM-2 parameters
model.eval()
for param in model.parameters():
  param.requires_grad = False

print("[+] Model loaded and weights frozen successfully.")

[+] Loading facebook/esm2_t33_650M_UR50D...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.61GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/534 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[+] Model loaded and weights frozen successfully.


In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


def extract_esm_features(
    csv_path: str, batch_size: int = 64, max_len: int = 128
):
  df = pd.read_csv(csv_path)
  sequences = df["seq"].astype(str).tolist()
  labels = torch.tensor(df["label"].values, dtype=torch.long)

  all_pooled = []
  all_tokens = []
  all_masks = []

  print(
      f"Processing {len(sequences)} sequences from {os.path.basename(csv_path)}..."
  )

  for i in tqdm(range(0, len(sequences), batch_size)):
    batch_seqs = sequences[i : i + batch_size]

    # Tokenize batch (padding up to batch max or max_len)
    encoded = tokenizer(
        batch_seqs,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
      outputs = model(input_ids=input_ids, attention_mask=attention_mask)
      # Shape: (batch_size, seq_len, 320)
      token_representations = outputs.last_hidden_state

      # ESM-2 adds <cls> at index 0 and <eos> at the end.
      # For mean-pooling, calculate average across non-special residue tokens:
      mask_expanded = (
          attention_mask.unsqueeze(-1).expand(token_representations.size()).float()
      )
      sum_embeddings = torch.sum(token_representations * mask_expanded, dim=1)
      sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
      mean_pooled = sum_embeddings / sum_mask

    # Transfer batch to CPU (float16 to optimize storage)
    all_tokens.append(token_representations.half().cpu())
    all_masks.append(attention_mask.bool().cpu())
    all_pooled.append(mean_pooled.half().cpu())

  # Concatenate along batch dimension with uniform padding
  final_pooled = torch.cat(all_pooled, dim=0)

  # Pad tokens across different batches to a uniform max sequence length
  max_seq_len = max(t.size(1) for t in all_tokens)
  padded_tokens = []
  padded_masks = []

  for tokens, masks in zip(all_tokens, all_masks):
    b, s, d = tokens.shape
    if s < max_seq_len:
      pad_t = torch.zeros(
          (b, max_seq_len - s, d), dtype=tokens.dtype, device=tokens.device
      )
      pad_m = torch.zeros(
          (b, max_seq_len - s), dtype=masks.dtype, device=masks.device
      )
      tokens = torch.cat([tokens, pad_t], dim=1)
      masks = torch.cat([masks, pad_m], dim=1)
    padded_tokens.append(tokens)
    padded_masks.append(masks)

  final_tokens = torch.cat(padded_tokens, dim=0)
  final_masks = torch.cat(padded_masks, dim=0)

  return {
      "pooled": final_pooled,  # (N, 320)
      "tokens": final_tokens,  # (N, L_max, 320)
      "masks": final_masks,  # (N, L_max)
      "labels": labels,  # (N,)
  }

In [ ]:
# 1. Extract Training Set Features
train_features = extract_esm_features(train_csv, batch_size=16)
torch.save(
    train_features, os.path.join(OUTPUT_DIR, "esm2_train_representations_largeModel.pt")
)
print("[+] Saved training set tensors to Drive.")

# 2. Extract Independent Test Set Features
test_features = extract_esm_features(test_csv, batch_size=16)
torch.save(
    test_features, os.path.join(OUTPUT_DIR, "esm2_test_representations_largeModel.pt")
)
print("[+] Saved test set tensors to Drive.")

print("\n" + "=" * 60)
print("PHASE 1 COMPLETE: All tensors saved to your Google Drive!")
print("=" * 60)

Processing 8038 sequences from train.csv...


  0%|          | 0/503 [00:00<?, ?it/s]

[+] Saved training set tensors to Drive.
Processing 888 sequences from test.csv...


  0%|          | 0/56 [00:00<?, ?it/s]

[+] Saved test set tensors to Drive.

PHASE 1 COMPLETE: All tensors saved to your Google Drive!


In [ ]:
print(f"Train Tokens Shape: {train_features['tokens'].shape}")
print(f"Train Pooled Shape: {train_features['pooled'].shape}")
print(f"Test Tokens Shape:  {test_features['tokens'].shape}")
print(f"Test Pooled Shape:  {test_features['pooled'].shape}")

Train Tokens Shape: torch.Size([8038, 101, 1280])
Train Pooled Shape: torch.Size([8038, 1280])
Test Tokens Shape:  torch.Size([888, 100, 1280])
Test Pooled Shape:  torch.Size([888, 1280])
